# Vesuvius Scroll Segment Visualizer

This notebook loads and visualizes the downloaded PHercParis4 scroll segments.

**Data layout**
```
data/scroll/
  {segment_id}/
    surface_volume/
      00.tif ... 64.tif   ← 65 z-slices (TIFF)
    mask.png               ← binary mask
    meta.json              ← segment metadata
```

Run all cells top-to-bottom. Adjust the `SEGMENT_ID` variable to switch segments.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tifffile as tiff

# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------
DATA_ROOT = Path("data/scroll")

# Choose which segment to explore (change this string to switch segments)
SEGMENT_ID = "20230827161847"   # Grand Prize — first readable text
# Other options:
# SEGMENT_ID = "20230826170124"
# SEGMENT_ID = "20230820174948"
# SEGMENT_ID = "20230826135043"
# SEGMENT_ID = "20230828154913"
# SEGMENT_ID = "20230819093803"
# SEGMENT_ID = "20230504093154"

seg_dir   = DATA_ROOT / SEGMENT_ID
sv_dir    = seg_dir / "surface_volume"
mask_path = seg_dir / "mask.png"
meta_path = seg_dir / "meta.json"

print(f"Segment directory: {seg_dir.resolve()}")
print(f"Layers dir       : {sv_dir.resolve()}")
print(f"Mask             : {mask_path.resolve()}")
print(f"Meta             : {meta_path.resolve()}")

## 1. Load metadata & mask

In [ ]:
# --- Metadata -------------------------------------------------------
with open(meta_path) as f:
    meta = json.load(f)
print("Metadata:")
for k, v in meta.items():
    print(f"  {k}: {v}")

# --- Mask -----------------------------------------------------------
mask = np.array(Image.open(mask_path).convert("L"))
mask_bool = mask > 128   # threshold if mask is grayscale

print(f"\nMask shape : {mask.shape}")
print(f"Mask dtype : {mask.dtype}")
print(f"Mask unique: {np.unique(mask)}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(mask_bool, cmap="gray")
ax.set_title(f"{SEGMENT_ID} — mask")
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Load a single z-slice (2-D image)

In [ ]:
z = 32   # middle layer (0 – 64)
slice_path = sv_dir / f"{z:02d}.tif"

img = tiff.imread(str(slice_path))
print(f"Layer {z:02d} shape : {img.shape}")
print(f"Layer {z:02d} dtype : {img.dtype}")
print(f"Layer {z:02d} range : [{img.min()}, {img.max()}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Raw slice
axes[0].imshow(img, cmap="gray")
axes[0].set_title(f"Layer {z:02d} — raw")
axes[0].axis("off")

# Masked slice (set outside-mask to NaN so it shows as transparent/white)
img_masked = img.astype(float)
img_masked[~mask_bool] = np.nan
axes[1].imshow(img_masked, cmap="gray")
axes[1].set_title(f"Layer {z:02d} — masked")
axes[1].axis("off")

# Histogram of pixel intensities inside mask
axes[2].hist(img[mask_bool].ravel(), bins=256, color="steelblue", edgecolor="none")
axes[2].set_title("Intensity histogram (inside mask)")
axes[2].set_xlabel("Intensity")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 3. Interactive z-stack viewer (all 65 layers)

In [ ]:
from ipywidgets import interact, IntSlider

@interact(z=IntSlider(min=0, max=64, step=1, value=32, description="Layer"))
def view_slice(z=32):
    slice_path = sv_dir / f"{z:02d}.tif"
    img = tiff.imread(str(slice_path))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(img, cmap="gray", vmin=img.min(), vmax=img.max())
    axes[0].set_title(f"Layer {z:02d} — raw")
    axes[0].axis("off")
    
    img_masked = img.astype(float)
    img_masked[~mask_bool] = np.nan
    axes[1].imshow(img_masked, cmap="gray", vmin=img.min(), vmax=img.max())
    axes[1].set_title(f"Layer {z:02d} — masked")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 4. Build a small 3-D volume (sub-sample for speed)

In [ ]:
# Load every Nth layer to keep memory low, or load all if you have RAM.
step = 1          # set to 2 or 4 to sub-sample layers Z-axis
downsample = 4    # spatial downsample factor (1 = full resolution)

layer_files = sorted(sv_dir.glob("*.tif"))[::step]
print(f"Loading {len(layer_files)} layers (step={step})...")

layers = []
for fp in layer_files:
    # Read the TIFF
    im = tiff.imread(str(fp))
    if downsample > 1:
        # Use .copy() to ensure we don't keep the original 92MB array in memory!
        im = im[::downsample, ::downsample].copy()
    layers.append(im)

volume = np.stack(layers, axis=0)   # shape: (Z, Y, X)
print(f"Volume shape : {volume.shape}")
print(f"Volume dtype : {volume.dtype}")
print(f"Volume size  : {volume.nbytes / 1e6:.1f} MB")


### 4.1 Orthogonal projections (MIP + mean)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Maximum Intensity Projection
axes[0, 0].imshow(volume.max(axis=0), cmap="gray")
axes[0, 0].set_title("MIP — axial (Z max)")
axes[0, 0].axis("off")

axes[0, 1].imshow(volume.max(axis=1), cmap="gray")
axes[0, 1].set_title("MIP — coronal (Y max)")
axes[0, 1].axis("off")

axes[0, 2].imshow(volume.max(axis=2), cmap="gray")
axes[0, 2].set_title("MIP — sagittal (X max)")
axes[0, 2].axis("off")

# Mean projection
axes[1, 0].imshow(volume.mean(axis=0), cmap="gray")
axes[1, 0].set_title("Mean — axial (Z mean)")
axes[1, 0].axis("off")

axes[1, 1].imshow(volume.mean(axis=1), cmap="gray")
axes[1, 1].set_title("Mean — coronal (Y mean)")
axes[1, 1].axis("off")

axes[1, 2].imshow(volume.mean(axis=2), cmap="gray")
axes[1, 2].set_title("Mean — sagittal (X mean)")
axes[1, 2].axis("off")

plt.suptitle(f"{SEGMENT_ID} — orthogonal projections (downsample={downsample})", fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 3-D surface rendering (simple marching cubes)

Requires `scikit-image`. Skip if you only need 2-D views.

In [ ]:
from skimage import measure

# Pick an intensity threshold for the isosurface
threshold = np.percentile(volume, 50)
print(f"Isosurface threshold (50th percentile): {threshold:.1f}")

verts, faces, normals, values = measure.marching_cubes(volume, level=threshold)
print(f"Vertices: {verts.shape[0]}, Faces: {faces.shape[0]}")

In [ ]:
# Matplotlib 3-D mesh plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")

# Sub-sample faces for faster rendering if mesh is huge
face_step = max(1, faces.shape[0] // 50_000)

ax.plot_trisurf(
    verts[:, 0], verts[:, 1], verts[:, 2],
    triangles=faces[::face_step],
    cmap="viridis",
    lw=0,
    alpha=0.8,
)
ax.set_title(f"{SEGMENT_ID} — isosurface @ {threshold:.1f}")
ax.set_xlabel("Z")
ax.set_ylabel("Y")
ax.set_zlabel("X")
plt.show()

## 5. Compare multiple segments side-by-side

In [ ]:
SEGMENTS = [
    "20230827161847",
    "20230826170124",
    "20230820174948",
    "20230826135043",
    "20230828154913",
    "20230819093803",
    "20230504093154",
]

z = 32
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for ax, seg_id in zip(axes, SEGMENTS):
    fp = DATA_ROOT / seg_id / "surface_volume" / f"{z:02d}.tif"
    im = tiff.imread(str(fp))
    ax.imshow(im, cmap="gray")
    ax.set_title(seg_id, fontsize=9)
    ax.axis("off")

# hide unused subplots
for ax in axes[len(SEGMENTS):]:
    ax.axis("off")

plt.suptitle(f"Layer {z:02d} across all segments", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Layer-wise statistics

In [ ]:
means = []
stds  = []
mins  = []
maxs  = []

for z in range(65):
    fp = sv_dir / f"{z:02d}.tif"
    im = tiff.imread(str(fp))
    means.append(im.mean())
    stds.append(im.std())
    mins.append(im.min())
    maxs.append(im.max())

means = np.array(means)
stds  = np.array(stds)
mins  = np.array(mins)
maxs  = np.array(maxs)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(means, marker="o", ms=3)
axes[0, 0].set_title("Mean intensity per layer")
axes[0, 0].set_xlabel("Layer")
axes[0, 0].set_ylabel("Mean")
axes[0, 0].grid(True, ls="--", alpha=0.4)

axes[0, 1].plot(stds, marker="o", ms=3, color="orange")
axes[0, 1].set_title("Std-dev per layer")
axes[0, 1].set_xlabel("Layer")
axes[0, 1].set_ylabel("Std")
axes[0, 1].grid(True, ls="--", alpha=0.4)

axes[1, 0].fill_between(range(65), mins, maxs, color="green", alpha=0.3)
axes[1, 0].plot(mins, color="green", lw=1, label="min")
axes[1, 0].plot(maxs, color="green", lw=1, label="max")
axes[1, 0].set_title("Min / Max per layer")
axes[1, 0].set_xlabel("Layer")
axes[1, 0].set_ylabel("Intensity")
axes[1, 0].legend()
axes[1, 0].grid(True, ls="--", alpha=0.4)

axes[1, 1].hist(means, bins=20, color="steelblue", edgecolor="white")
axes[1, 1].set_title("Distribution of layer means")
axes[1, 1].set_xlabel("Mean intensity")
axes[1, 1].set_ylabel("Count")

plt.suptitle(f"{SEGMENT_ID} — layer-wise statistics", fontsize=14)
plt.tight_layout()
plt.show()

---
**Tips**
- Change `SEGMENT_ID` at the top to explore other segments.
- Increase `downsample` in section 4 if you run out of RAM.
- Use the interactive slider in section 3 to quickly scan layers.
- Adjust the `threshold` in section 4.2 to get cleaner / noisier surfaces.

## 7. Additional Visualizations: Ink Depth-Map
Let's visualize the Z-depth of the maximum intensity pixels to see the geometry of the scroll.

In [ ]:
# Let's find the Z-index of the maximum intensity for each pixel (where the ink/brightest object is)
z_indices = np.argmax(volume, axis=0)
mip = np.max(volume, axis=0)

# We can mask this by thresholding the MIP so we only show the depth for pixels with some structure
threshold = np.percentile(mip, 80)
depth_map = np.where(mip > threshold, z_indices, np.nan)

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
cmap = plt.cm.jet
cmap.set_bad(color='black')
im = ax.imshow(depth_map, cmap=cmap)
plt.colorbar(im, ax=ax, label='Z layer index of Maximum Intensity')
ax.set_title(f"{SEGMENT_ID} — Depth Map of Brightest Structures")
ax.axis("off")
plt.tight_layout()
plt.show()


## 8. Additional Visualizations: 3D Visualization of the Depth Map

In [ ]:
# We can visualize the depth map in 3D to see the curved surface of the scroll!
# To save plotting time, we downsample it further.
D3_DOWNSAMPLE = 4
depth_small = depth_map[::D3_DOWNSAMPLE, ::D3_DOWNSAMPLE]
X = np.arange(depth_small.shape[1])
Y = np.arange(depth_small.shape[0])
X, Y = np.meshgrid(X, Y)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
# Surface plot, mask out NaNs
surf = ax.plot_surface(X, Y, depth_small, cmap='jet', linewidth=0, antialiased=False)
ax.set_title("3D Surface Profile of Scroll Intensity")
ax.view_init(elev=30, azim=45)
plt.show()
